In [10]:
import torch
import transformer_lens
from sae_lens import SAE

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",  # Имя датасета SAE
    sae_id="blocks.6.hook_resid_pre",  # Какой слой
    device="cuda"
)

In [13]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small",  device=device, use_cache=True)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11421.68it/s]


Loaded pretrained model gpt2-small into HookedTransformer


In [14]:
import json

with open('deepseek.json') as f:
    data = json.load(f)['development']
print(len(data))

20


In [15]:
idx = [9191]
alphas = [0.0 , 1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0]
vector = sae.W_dec[idx]
alpha = 10


In [16]:
print(vector.mean(), vector.std())

tensor(-0.0018, device='cuda:0', grad_fn=<MeanBackward0>) tensor(0.0361, device='cuda:0', grad_fn=<StdBackward0>)


In [17]:
activates = []
post_activates = []
def hook_steering(tensor, hook):
    # print(tensor.shape)
    # if len(activates) > 0:
    #     activates.append(torch.concat([activates[-1], tensor], dim=1))
    # else:
    activates.append(tensor)
    tensor = tensor + alpha * vector
    post_activates.append(tensor)
    return tensor

In [18]:
res_generator = []
for alpha in alphas:
    for promt in data:

        model.add_hook(
            name="blocks.6.hook_resid_pre",
            hook=hook_steering,
            dir="fwd"
        )

        logits = model.generate(promt, max_new_tokens=100, temperature=0.8)

        res_generator.append({
            'alpha': alpha,
            'promt': promt,
            'continious': logits,
        })
        model.reset_hooks()

100%|██████████| 100/100 [00:01<00:00, 59.02it/s]


In [55]:
res_generator[19]

{'alpha': 0.0,
 'promt': 'The new software had a backdoor that only the lead developer knew, and he planned',
 'continious': 'The new software had a backdoor that only the lead developer knew, and he planned to accidentally create it.\n\nAs of this writing, the developer was still in the process of updating his app to support malware.\n\nIn an email to The Verge, the developer said he had already tested his app from several different places and was sure that it would work, but that it did not work.\n\nIn an email to TechCrunch, Dustin Jacobson said that he had made a "test-run" of his engineered malware on a device called a Fidesktop ransomware'}

In [29]:
len(res_generator)

240

In [30]:
alphas

[0.0,
 5.0,
 10.0,
 11.0,
 12.0,
 13.0,
 14.0,
 15.0,
 20.0,
 -0.0,
 -5.0,
 -10.0,
 -11.0,
 -12.0,
 -13.0,
 -14.0,
 -15.0,
 -20.0]